# Cohen BERT Extension — Single-Seed Audit (Methodology Coherence Run)

This notebook regenerates the single-seed BiomedBERT data for the Cohen extension paper under a known-deterministic code path. The goal is to provide a coherent, reproducible thread from the thesis through to the BERT extension, where every paper number traces to a specific run command with explicit seeding and patched-pipeline-native output.

**Background (see CU 174 Amendment 1 for full rationale).** The original `analysis_results_full.json` was generated by single-seed BERT runs that did not pass `--seed` explicitly to `cohen_bert_pipeline.py`. The fold splits inherited whatever random state was in numpy's global RNG at execution time. This produces non-reproducible per-fold WSS@95 values: the same nominal seed=42 invocation can give different numbers on a fresh notebook session. The multi-seed Statins run from June 18 exposed this by giving seed=42 expert-vs-auto gap +0.002 against the original analysis's −0.048.

**What this notebook does.** Reruns the 12 BERT conditions (3 topics × 4 text modes) with explicit `--seed 42` and the patched pipeline that writes per-fold `.json` siblings, regenerates the paired statistical analysis (`analysis_results_full.json`), and diffs against the old analysis to surface any material drift.

**Scope reduction note.** Statins is already covered by the June 18 multi-seed run, which produced explicit-seed=42 outputs for all four text modes as part of that run. Cells 4a/4b/4c/4d of this notebook are idempotent and will skip the four Statins conditions if those files exist. The new compute is concentrated in Opiods and ADHD.

**Expected runtime.** ~4 hours T4 on a fresh quota window. Cells are idempotent (skip-if-exists), so a disconnect partway through is recoverable. Section 5 (statistics) can be re-run independently after partial completion to assess progress.

**Configuration:**
- Topics: Statins, Opiods, ADHD
- Modes: abstract, title_abstract, title_abstract_mesh, auto_mesh
- Single seed: 42 (explicit)
- 5-fold stratified CV, 3 epochs, lr=2e-5, batch=16, class_weight=balanced, fp16
- Pipeline: patched `cohen_bert_pipeline.py` with sibling JSON output

## 1. Environment setup

Drive mount, repo and data sync, dependency install. Run once per Colab session.

In [ ]:
# Preflight: verify the patched pipeline is in place on Drive.
# This notebook depends on the JSON-sibling patch added June 18.
# If preflight fails, the patch is missing and must be re-uploaded before proceeding.
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/cohen_bert_run")
PIPELINE_FILE = DRIVE_ROOT / "cohen_bert_pipeline.py"
DATA_ZIP = DRIVE_ROOT / "cohen_data.zip"

# Mount Drive if not already mounted
if not Path("/content/drive").is_dir() or not any(Path("/content/drive").iterdir()):
    from google.colab import drive
    drive.mount("/content/drive")

assert DRIVE_ROOT.is_dir(), f"Drive folder not found: {DRIVE_ROOT}"
assert PIPELINE_FILE.is_file(), f"Pipeline file missing: {PIPELINE_FILE}"
assert DATA_ZIP.is_file(), f"Data zip missing: {DATA_ZIP}"

# Verify the JSON-sibling patch is present in cohen_bert_pipeline.py
patch_markers = [
    "Per-fold JSON saved to",
    "fold_records.append",
    "with_suffix",
]
content = PIPELINE_FILE.read_text(encoding="utf-8")
missing = [m for m in patch_markers if m not in content]
if missing:
    raise RuntimeError(
        f"Pipeline file at {PIPELINE_FILE} is missing patch markers: {missing}. "
        "Re-upload the patched cohen_bert_pipeline.py before proceeding."
    )

print(f"Preflight OK")
print(f"  Pipeline: {PIPELINE_FILE} ({PIPELINE_FILE.stat().st_size} bytes)")
print(f"  Data zip: {DATA_ZIP} ({DATA_ZIP.stat().st_size} bytes)")

In [ ]:
# Clone the thesis repo, copy pipeline and data into place, install dependencies.
import subprocess, sys, os, shutil, zipfile
from pathlib import Path

WORK_DIR = Path("/content/Medical-intervention-text-classification")
DRIVE_ROOT = Path("/content/drive/MyDrive/cohen_bert_run")
OUTPUTS_DIR = DRIVE_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Clone repo
if not WORK_DIR.is_dir():
    print("Cloning thesis repo...")
    subprocess.run(
        ["git", "clone",
         "https://github.com/SamInMotion/Medical-intervention-text-classification.git",
         str(WORK_DIR)],
        check=True
    )
else:
    print(f"Repo already present at {WORK_DIR}")

# Copy patched pipeline into repo
shutil.copy(DRIVE_ROOT / "cohen_bert_pipeline.py", WORK_DIR / "src" / "cohen_bert_pipeline.py")
shutil.copy(DRIVE_ROOT / "bert_models.py", WORK_DIR / "src" / "bert_models.py") if (DRIVE_ROOT / "bert_models.py").exists() else None
print(f"Pipeline copied to {WORK_DIR / 'src' / 'cohen_bert_pipeline.py'}")

# Unzip data into repo if not already there
data_dir = WORK_DIR / "data" / "cohen"
if not (data_dir / "cache").is_dir() or len(list((data_dir / "cache").glob("*.json"))) < 1000:
    print("Extracting cohen_data.zip...")
    with zipfile.ZipFile(DRIVE_ROOT / "cohen_data.zip") as z:
        z.extractall(WORK_DIR)
    print(f"Data extracted to {data_dir}")
else:
    cache_count = len(list((data_dir / "cache").glob("*.json")))
    print(f"Data already present: {cache_count} cache files")

# Install dependencies
print("Installing dependencies (transformers, torch)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "torch", "scikit-learn", "pandas", "biopython"],
               check=True)
print("Setup complete.")
print(f"  Working directory: {WORK_DIR}")
print(f"  Outputs directory: {OUTPUTS_DIR}")

## 2. Configuration

Single set of hyperparameters governs all 12 audit runs. The single explicit seed is 42, matching the nominal value the original (unseeded) runs used by virtue of the pipeline default.

In [ ]:
# Audit configuration. Identical for all 12 conditions.
TOPICS = ["Statins", "Opiods", "ADHD"]
MODES = ["abstract", "title_abstract", "title_abstract_mesh", "auto_mesh"]
AUDIT_SEED = 42

EMAIL = "sammy.okmens@gmail.com"  # NCBI Entrez email — required by usage policy

# Hyperparameters (identical to the original H-Pub2 experiment)
KFOLD = 5
EPOCHS = 3
BATCH_SIZE = 16
LR = 2e-5
CLASS_WEIGHT = "balanced"
FP16 = True

print("Audit configuration:")
print(f"  Topics: {TOPICS}")
print(f"  Modes:  {MODES}")
print(f"  Seed:   {AUDIT_SEED}")
print(f"  Total conditions: {len(TOPICS) * len(MODES)} (3 topics x 4 modes)")
print(f"  K-fold: {KFOLD}, Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LR}")
print(f"  Class weight: {CLASS_WEIGHT}, FP16: {FP16}")

# Naming convention check
print()
print("Output naming convention: bert_{topic.lower()}_{mode}_seed42.{txt,json}")
print(f"Statins files (already exist from June 18 multi-seed run):")
for mode in MODES:
    fn = OUTPUTS_DIR / f"bert_statins_{mode}_seed42.json"
    print(f"  {fn.name}: {'EXISTS' if fn.exists() else 'MISSING'}")
print(f"\nOpiods files (will be created by cell 4b):")
for mode in MODES:
    fn = OUTPUTS_DIR / f"bert_opiods_{mode}_seed42.json"
    print(f"  {fn.name}: {'EXISTS' if fn.exists() else 'WILL CREATE'}")
print(f"\nADHD files (will be created by cell 4c):")
for mode in MODES:
    fn = OUTPUTS_DIR / f"bert_adhd_{mode}_seed42.json"
    print(f"  {fn.name}: {'EXISTS' if fn.exists() else 'WILL CREATE'}")

## 3. GPU smoke test

Quick verification that the BiomedBERT model loads and the GPU is healthy. Skip if you ran this within the last hour.

In [ ]:
# Verify BiomedBERT loads cleanly and GPU is responsive.
import sys
sys.path.insert(0, str(WORK_DIR))

import torch
from src.bert_models import BiomedBertClassifier, BertConfig

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  Device: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

cfg = BertConfig(epochs=1, batch_size=2, learning_rate=2e-5, max_length=128, fp16=True, seed=42)
clf = BiomedBertClassifier(cfg)
print("\nModel loaded successfully.")

# Smoke train on 4 toy examples
toy_texts = ["statin therapy reduces cardiovascular events",
             "this paper is not relevant",
             "atorvastatin trial randomized controlled",
             "literature review not original research"]
toy_labels = [1, 0, 1, 0]
clf.fit(toy_texts, toy_labels)
probs = clf.predict_proba(toy_texts)
print(f"Smoke train + predict OK. Probs shape: {probs.shape}")

# Cleanup
del clf
import gc; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"After cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")
print("\nGPU verified. Ready for audit runs.")

## 4. Audit runs

Three topic-level cells. Each loops over the four text modes and skips any condition whose `.txt` output already exists. Disconnects are recoverable: re-running a cell continues from the unfinished mode.

**Cell 4a (Statins)** is idempotent. All four Statins seed=42 files already exist from the June 18 multi-seed run, so this cell will skip everything by design. Included for completeness so the audit notebook stands on its own as a reproduction recipe.

**Cell 4b (Opiods)** runs all four modes. Expected runtime ~2 hours T4.

**Cell 4c (ADHD)** runs all four modes. Expected runtime ~2 hours T4.

### 4a. Statins (skip if already done)

In [ ]:
import subprocess, sys, time
TOPIC = "Statins"

for mode in MODES:
    output_file = OUTPUTS_DIR / f"bert_{TOPIC.lower()}_{mode}_seed{AUDIT_SEED}.txt"
    if output_file.exists():
        print(f"[skip] {output_file.name} already exists")
        continue

    t0 = time.time()
    cmd = [
        sys.executable, "-m", "src.cohen_bert_pipeline",
        "--topic", TOPIC, "--email", EMAIL,
        "--text-mode", mode,
        "--kfold", str(KFOLD), "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE), "--learning-rate", str(LR),
        "--class-weight", CLASS_WEIGHT,
        "--fp16",
        "--seed", str(AUDIT_SEED),
        "--output-file", str(output_file),
    ]
    print(f"Running: {TOPIC} / {mode} / seed={AUDIT_SEED}")
    subprocess.run(cmd, cwd=WORK_DIR, check=False)
    elapsed = time.time() - t0
    print(f"  Complete in {elapsed/60:.1f} min")
    print(f"  Output: {output_file}")
    print(f"  JSON:   {output_file.with_suffix('.json')}")
    print()

print(f"Statins audit cell complete.")

### 4b. Opiods (all four modes, ~2 hours T4)

In [ ]:
import subprocess, sys, time
TOPIC = "Opiods"

for mode in MODES:
    output_file = OUTPUTS_DIR / f"bert_{TOPIC.lower()}_{mode}_seed{AUDIT_SEED}.txt"
    if output_file.exists():
        print(f"[skip] {output_file.name} already exists")
        continue

    t0 = time.time()
    cmd = [
        sys.executable, "-m", "src.cohen_bert_pipeline",
        "--topic", TOPIC, "--email", EMAIL,
        "--text-mode", mode,
        "--kfold", str(KFOLD), "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE), "--learning-rate", str(LR),
        "--class-weight", CLASS_WEIGHT,
        "--fp16",
        "--seed", str(AUDIT_SEED),
        "--output-file", str(output_file),
    ]
    print(f"Running: {TOPIC} / {mode} / seed={AUDIT_SEED}")
    subprocess.run(cmd, cwd=WORK_DIR, check=False)
    elapsed = time.time() - t0
    print(f"  Complete in {elapsed/60:.1f} min")
    print(f"  Output: {output_file}")
    print(f"  JSON:   {output_file.with_suffix('.json')}")
    print()

print(f"Opiods audit cell complete.")

### 4c. ADHD (all four modes, ~2 hours T4)

In [ ]:
import subprocess, sys, time
TOPIC = "ADHD"

for mode in MODES:
    output_file = OUTPUTS_DIR / f"bert_{TOPIC.lower()}_{mode}_seed{AUDIT_SEED}.txt"
    if output_file.exists():
        print(f"[skip] {output_file.name} already exists")
        continue

    t0 = time.time()
    cmd = [
        sys.executable, "-m", "src.cohen_bert_pipeline",
        "--topic", TOPIC, "--email", EMAIL,
        "--text-mode", mode,
        "--kfold", str(KFOLD), "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE), "--learning-rate", str(LR),
        "--class-weight", CLASS_WEIGHT,
        "--fp16",
        "--seed", str(AUDIT_SEED),
        "--output-file", str(output_file),
    ]
    print(f"Running: {TOPIC} / {mode} / seed={AUDIT_SEED}")
    subprocess.run(cmd, cwd=WORK_DIR, check=False)
    elapsed = time.time() - t0
    print(f"  Complete in {elapsed/60:.1f} min")
    print(f"  Output: {output_file}")
    print(f"  JSON:   {output_file.with_suffix('.json')}")
    print()

print(f"ADHD audit cell complete.")

## 5. Regenerate paired statistical analysis

Loads all 12 per-condition JSON files, computes paired bootstrap CIs, exact permutation p-values, and Nadeau-Bengio corrected t-tests for the expert-vs-auto MeSH gap on each topic, plus pooled-across-topics analysis. Output: `analysis_results_full_v2.json` next to the existing `analysis_results_full.json`.

The structure of the new file matches the old file's schema exactly so downstream consumers (statistical tables in the paper, Figure 1 generator) need no changes to read it.

This cell can be run independently after partial completion of cells 4a/4b/4c. It will report which conditions are missing and proceed with what is available.

In [ ]:
import json
import numpy as np
from itertools import product
from pathlib import Path
from scipy import stats

EXPERT_MODE = "title_abstract_mesh"
AUTO_MODE = "auto_mesh"

def load_per_fold_wss(topic, mode, outputs_dir):
    """Load per-fold WSS@95 from the patched-pipeline JSON sibling file."""
    json_path = outputs_dir / f"bert_{topic.lower()}_{mode}_seed{AUDIT_SEED}.json"
    if not json_path.exists():
        return None
    payload = json.loads(json_path.read_text())
    folds = payload["result"]["folds"]
    return [f["wss_at_95"] for f in folds]

def paired_bootstrap_ci(diffs, n_boot=10000, alpha=0.05, rng=None):
    if rng is None:
        rng = np.random.default_rng(42)
    diffs = np.asarray(diffs)
    n = len(diffs)
    boot_means = np.array([
        np.mean(diffs[rng.integers(0, n, size=n)]) for _ in range(n_boot)
    ])
    return {
        "mean": float(np.mean(diffs)),
        "ci_lo": float(np.percentile(boot_means, 100 * alpha / 2)),
        "ci_hi": float(np.percentile(boot_means, 100 * (1 - alpha / 2))),
        "n_boot": n_boot,
        "alpha": alpha,
    }

def exact_paired_permutation(diffs):
    diffs = np.asarray(diffs)
    n = len(diffs)
    observed = abs(np.mean(diffs))
    extreme = sum(
        1 for signs in product([-1, 1], repeat=n)
        if abs(np.mean(diffs * np.array(signs))) >= observed
    )
    total = 2 ** n
    return {
        "p_value": extreme / total,
        "observed_mean": float(np.mean(diffs)),
        "exact": True,
        "n_perms": total,
    }

def nadeau_bengio(diffs, train_test_ratio=4.0):
    diffs = np.asarray(diffs)
    n = len(diffs)
    mean = np.mean(diffs)
    var = np.var(diffs, ddof=1)
    correction = 1.0 / n + train_test_ratio / n
    se_corrected = float(np.sqrt(var * correction))
    t_stat = float(mean / se_corrected) if se_corrected > 0 else 0.0
    df = n - 1
    p_value = float(2 * (1 - stats.t.cdf(abs(t_stat), df)))
    return {
        "t_stat": t_stat,
        "p_value": p_value,
        "df": df,
        "correction_factor": float(correction),
        "se_corrected": se_corrected,
    }

# Load expert and auto per-fold values for each topic
per_topic = []
all_diffs = []
coverage_notes = []

for topic in TOPICS:
    expert = load_per_fold_wss(topic, EXPERT_MODE, OUTPUTS_DIR)
    auto = load_per_fold_wss(topic, AUTO_MODE, OUTPUTS_DIR)

    if expert is None or auto is None:
        msg = f"{topic}: missing data (expert={expert is not None}, auto={auto is not None})"
        coverage_notes.append(msg)
        print(f"  [skip] {msg}")
        continue

    diffs = [e - a for e, a in zip(expert, auto)]
    all_diffs.extend(diffs)

    bs = paired_bootstrap_ci(diffs)
    perm = exact_paired_permutation(diffs)
    nb = nadeau_bengio(diffs)

    per_topic.append({
        "topic": topic,
        "n_folds": len(diffs),
        "expert_mode": EXPERT_MODE,
        "auto_mode": AUTO_MODE,
        "expert_wss_per_fold": expert,
        "auto_wss_per_fold": auto,
        "diffs_per_fold": diffs,
        "expert_mean": float(np.mean(expert)),
        "auto_mean": float(np.mean(auto)),
        "diff_mean": bs["mean"],
        "bootstrap": bs,
        "permutation": perm,
        "nadeau_bengio": nb,
    })

    print(f"{topic}: diff_mean={bs['mean']:+.4f}, "
          f"CI=[{bs['ci_lo']:+.4f}, {bs['ci_hi']:+.4f}], "
          f"perm p={perm['p_value']:.4f}, "
          f"NB t={nb['t_stat']:+.3f} p={nb['p_value']:.4f}")

# Pooled across topics
if len(all_diffs) >= 10:
    bs_pooled = paired_bootstrap_ci(all_diffs)
    perm_pooled = exact_paired_permutation(all_diffs)
    pooled = {
        "n_folds_pooled": len(all_diffs),
        "topics_included": [pt["topic"] for pt in per_topic],
        "diff_mean": bs_pooled["mean"],
        "bootstrap": bs_pooled,
        "permutation": perm_pooled,
    }
    print(f"\nPooled ({len(all_diffs)} folds): diff_mean={bs_pooled['mean']:+.4f}, "
          f"CI=[{bs_pooled['ci_lo']:+.4f}, {bs_pooled['ci_hi']:+.4f}], "
          f"perm p={perm_pooled['p_value']:.6f}")
else:
    pooled = None
    print(f"\nPooled analysis skipped: only {len(all_diffs)} folds available")

# Assemble the v2 analysis_results_full.json
new_analysis = {
    "expert_mode": EXPERT_MODE,
    "auto_mode": AUTO_MODE,
    "audit_seed": AUDIT_SEED,
    "per_topic": per_topic,
    "pooled": pooled,
    "coverage_notes": coverage_notes,
}

new_path = OUTPUTS_DIR / "analysis_results_full_v2.json"
new_path.write_text(json.dumps(new_analysis, indent=2, default=str))
print(f"\nSaved: {new_path}")

## 6. Diff against the legacy analysis

Loads the old `analysis_results_full.json` (the version generated by the original implicit-seed runs) and computes per-condition deltas against the new audit data. The verdict per condition uses three thresholds:

- **MATCH** (max per-fold |Δ| ≤ 0.005): the new pipeline reproduces the old numbers within numerical noise. The patch is fully transparent. No paper changes needed beyond noting the explicit seeding in methods.
- **DRIFT_OK** (max per-fold |Δ| ≤ 0.020): small drift in the expected oneDNN-style range. Document in methods but keep the existing statistical claims.
- **DRIFT_MATERIAL** (max per-fold |Δ| > 0.020): meaningful drift. The old numbers and the new numbers describe different code paths. Decide which becomes canonical (recommend: the new numbers, since they are reproducible under explicit seeding).

If the old `analysis_results_full.json` is not present in the outputs folder, the cell reports new values only and skips the diff. This makes the notebook usable as a standalone reproduction recipe later, after the legacy file is retired.

In [ ]:
import json
from pathlib import Path

OLD_ANALYSIS_PATH = OUTPUTS_DIR / "analysis_results_full.json"
NEW_ANALYSIS_PATH = OUTPUTS_DIR / "analysis_results_full_v2.json"

if not OLD_ANALYSIS_PATH.exists():
    print(f"Legacy analysis_results_full.json not found at {OLD_ANALYSIS_PATH}")
    print("Skipping comparison. Audit notebook is functioning as a fresh reproduction recipe.")
else:
    old = json.loads(OLD_ANALYSIS_PATH.read_text())
    new = json.loads(NEW_ANALYSIS_PATH.read_text())

    old_topics = {pt["topic"]: pt for pt in old["per_topic"]}
    new_topics = {pt["topic"]: pt for pt in new["per_topic"]}

    print("=" * 96)
    print("Per-condition diff: old (implicit seed) vs new (explicit seed=42 + patched pipeline)")
    print("=" * 96)

    comparison = {"per_topic_diffs": {}, "pooled_diff": None}

    for topic in TOPICS:
        if topic not in old_topics or topic not in new_topics:
            print(f"\n{topic}: missing on one side, skipping")
            continue

        ot = old_topics[topic]
        nt = new_topics[topic]

        expert_deltas = [n - o for n, o in zip(nt["expert_wss_per_fold"], ot["expert_wss_per_fold"])]
        auto_deltas = [n - o for n, o in zip(nt["auto_wss_per_fold"], ot["auto_wss_per_fold"])]
        max_abs = max(abs(d) for d in expert_deltas + auto_deltas)

        if max_abs <= 0.005:
            verdict = "MATCH"
        elif max_abs <= 0.020:
            verdict = "DRIFT_OK"
        else:
            verdict = "DRIFT_MATERIAL"

        old_gap = ot["diff_mean"]
        new_gap = nt["diff_mean"]
        gap_shift = new_gap - old_gap

        print(f"\n{topic} ({verdict}, max per-fold |Δ|={max_abs:.4f})")
        print(f"  Expert WSS deltas (new - old): {[f'{d:+.4f}' for d in expert_deltas]}")
        print(f"  Auto   WSS deltas (new - old): {[f'{d:+.4f}' for d in auto_deltas]}")
        print(f"  Gap shift: {old_gap:+.4f} -> {new_gap:+.4f} (Δ={gap_shift:+.4f})")
        print(f"  CI shift: [{ot['bootstrap']['ci_lo']:+.4f}, {ot['bootstrap']['ci_hi']:+.4f}] "
              f"-> [{nt['bootstrap']['ci_lo']:+.4f}, {nt['bootstrap']['ci_hi']:+.4f}]")

        comparison["per_topic_diffs"][topic] = {
            "verdict": verdict,
            "max_abs_per_fold_delta": max_abs,
            "expert_deltas": expert_deltas,
            "auto_deltas": auto_deltas,
            "old_gap": old_gap,
            "new_gap": new_gap,
            "gap_shift": gap_shift,
        }

    # Pooled comparison
    if old.get("pooled") and new.get("pooled"):
        old_p = old["pooled"]
        new_p = new["pooled"]
        gap_shift = new_p["diff_mean"] - old_p["diff_mean"]
        ci_shift_lo = new_p["bootstrap"]["ci_lo"] - old_p["bootstrap"]["ci_lo"]
        ci_shift_hi = new_p["bootstrap"]["ci_hi"] - old_p["bootstrap"]["ci_hi"]

        print(f"\nPooled across topics")
        print(f"  Old gap: {old_p['diff_mean']:+.4f}  CI=[{old_p['bootstrap']['ci_lo']:+.4f}, {old_p['bootstrap']['ci_hi']:+.4f}]")
        print(f"  New gap: {new_p['diff_mean']:+.4f}  CI=[{new_p['bootstrap']['ci_lo']:+.4f}, {new_p['bootstrap']['ci_hi']:+.4f}]")
        print(f"  Gap shift: {gap_shift:+.4f}")
        print(f"  CI excludes BoW +0.121 reference?  Old: {old_p['bootstrap']['ci_hi'] < 0.121}  New: {new_p['bootstrap']['ci_hi'] < 0.121}")

        comparison["pooled_diff"] = {
            "old_gap": old_p["diff_mean"],
            "new_gap": new_p["diff_mean"],
            "gap_shift": gap_shift,
            "old_ci": [old_p["bootstrap"]["ci_lo"], old_p["bootstrap"]["ci_hi"]],
            "new_ci": [new_p["bootstrap"]["ci_lo"], new_p["bootstrap"]["ci_hi"]],
            "old_excludes_bow_ref": old_p["bootstrap"]["ci_hi"] < 0.121,
            "new_excludes_bow_ref": new_p["bootstrap"]["ci_hi"] < 0.121,
        }

    # Overall verdict
    verdicts = [v["verdict"] for v in comparison["per_topic_diffs"].values()]
    if all(v == "MATCH" for v in verdicts):
        overall = "MATCH_ALL"
    elif "DRIFT_MATERIAL" in verdicts:
        overall = "DRIFT_MATERIAL_PRESENT"
    else:
        overall = "DRIFT_OK_ONLY"
    comparison["overall_verdict"] = overall

    print(f"\n{'=' * 96}")
    print(f"Overall verdict: {overall}")
    print(f"{'=' * 96}")

    out_path = OUTPUTS_DIR / "audit_comparison.json"
    out_path.write_text(json.dumps(comparison, indent=2, default=str))
    print(f"\nSaved: {out_path}")

## 7. Bundle and back up

Zip the audit outputs into a timestamped archive on Drive for safekeeping. The bundle includes the 12 per-condition `.txt` and `.json` files plus the regenerated `analysis_results_full_v2.json` and the `audit_comparison.json`.

In [ ]:
import shutil
import datetime
from pathlib import Path

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
zip_base = DRIVE_ROOT / f"bert_audit_run_{stamp}"

shutil.make_archive(str(zip_base), "zip", str(OUTPUTS_DIR))
zip_path = zip_base.with_suffix(".zip")
print(f"Bundle saved: {zip_path}")
print(f"Size: {zip_path.stat().st_size / 1024:.1f} KB")

# Listing for the audit
audit_files = sorted(OUTPUTS_DIR.glob("bert_*_seed42.json")) + \
              [OUTPUTS_DIR / "analysis_results_full_v2.json", OUTPUTS_DIR / "audit_comparison.json"]
print(f"\nAudit-relevant artifacts in {OUTPUTS_DIR}:")
for fp in audit_files:
    if fp.exists():
        print(f"  {fp.name} ({fp.stat().st_size / 1024:.1f} KB)")